### Building Agents
* Part 1: A simple "Agent" and "Agent Loop". Basically an LLM call. We'll add tracing and streaming to the mix.
* Part 2: Adding a Tool.
* Part 3: Adding Memory

In [5]:
# The imports

import os
import requests
from dotenv import load_dotenv
from openai.types.responses import ResponseTextDeltaEvent
from agents import Agent, Runner, trace, function_tool, SQLiteSession
import agents
load_dotenv(override=True)


True

### What is an SDK?

An SDK (Software Development Kit) is a collection of tools, libraries, APIs, and documentation that helps developers build applications for a particular platform or service.

These classes and functions are provided by the OpenAI Agents SDK, whose PyPI package is typically named openai-agents.

In [6]:
print(Agent.__module__)
print(agents.__file__)

agents.agent
/Users/nareshchaurasia/nc/PYTHON-ARCHITECT/Python-Immersive-AI-MAC/RnD/Agents-Ed-Donner/.venv_ed_donner/lib/python3.12/site-packages/agents/__init__.py


In [7]:
from importlib.metadata import packages_distributions

print(packages_distributions()["agents"])

['openai-agents']


In [ ]:
import os
from dotenv import load_dotenv

load_dotenv("/Users/nareshchaurasia/nc/PYTHON-ARCHITECT/Python-Immersive-AI-MAC/.env_rag")

api_key = os.getenv("CO_API_KEY")
print(api_key)

In [ ]:
# This __replaces__ the default OpenAI-backend processor with LangSmith's, so traces get sent to your LangSmith project instead of `platform.openai.com/traces`.

from langsmith.wrappers import OpenAIAgentsTracingProcessor
from agents import set_trace_processors

set_trace_processors([OpenAIAgentsTracingProcessor()])

In [ ]:
from langsmith import Client
client = Client()
print(client.list_projects(limit=1))  # should not raise 401 if key is valid

In [ ]:
# import os
# print(repr(os.getenv("LANGSMITH_API_KEY")))
# print(repr(os.getenv("LANGSMITH_TRACING")))
# print(repr(os.getenv("LANGSMITH_PROJECT")))

In [ ]:

# Make an agent with name, instructions, model

agent = Agent(name="Jokester", instructions="You are a joke teller", model="gpt-5.4-mini")

In [ ]:
# Run the joke with Runner.run(agent, prompt)
# LangSmith - Agent workflow
result = await Runner.run(agent, "Tell a joke about Autonomous AI Agents")


In [ ]:
# Here is the final output

print(result.final_output)

In [ ]:
# Here is the detail of the LLM calls

result.to_input_list()

### Adding Observability with a trace

**Where traces are stored**

By default, traces from the OpenAI Agents SDK (`agents` package, used with `Runner.run`) are sent to __OpenAI's servers__ and viewable in the __OpenAI Platform dashboard__, specifically at:

__[](https://platform.openai.com/traces)<https://platform.openai.com/traces>__ (or similarly named "Traces"/"Logs" section under your OpenAI account)


In [ ]:
# LangSmith
with trace("Telling a joke"):
    result = await Runner.run(agent, "Tell a joke about Autonomous AI Agents")
print(result.final_output)

### Steaming

In [ ]:
# Streaming

result = Runner.run_streamed(agent, input="Please tell me 3 jokes about AI Agents.")
async for event in result.stream_events():
    if event.type == "raw_response_event" and isinstance(event.data, ResponseTextDeltaEvent):
        print(event.data.delta, end="", flush=True)

## Part 2: Adding a tool

In [ ]:
# pushover_user = os.getenv("PUSHOVER_USER")
# pushover_token = os.getenv("PUSHOVER_TOKEN")
# pushover_url = "https://api.pushover.net/1/messages.json"

# if pushover_user:
#     if pushover_user.startswith("u"):
#         print("Pushover user found and looks good")
#     else:
#         print("Pushover user found but doesn't start with u")
# else:
#     print("Pushover user not found")

# if pushover_token:
#     if pushover_token.startswith("a"):
#         print("Pushover token found and looks good")
#     else:
#         print("Pushover token found but doesn't start with a")
# else:
#     print("Pushover token not found")

In [ ]:
# Remember this?

def push(message):
    print(f"Push: {message}")
    # payload = {"user": pushover_user, "token": pushover_token, "message": message}
    # requests.post(pushover_url, data=payload)

In [ ]:
push("HEY!!")

In [ ]:
push

In [ ]:
# Now this:

@function_tool
def push_tool(message: str) -> str:
    # """ Send the given message to the user as a push notification """
    # payload = {"user": pushover_user, "token": pushover_token, "message": message}
    # result = requests.post(pushover_url, data=payload).status_code
    # return f"Push sent with API status code {result}"

    """ Send the given message to the user as a push notification """
    return f"Push: {message}"

In [ ]:
push_tool

In [ ]:
push_tool.description

In [ ]:

notifier = Agent(name="Notifier", model="gpt-5.4-mini", instructions="You notify the user upon request", tools=[push_tool])

In [ ]:
with trace("Pizza has arrived"):
    result = await Runner.run(notifier, "Notify the user that the pizza is here")

print(result.final_output)


## Now go and look at the trace

https://platform.openai.com/traces

## Part 3: Sessions (memory)

Within a Runner.run() application level turn, the conversation history is maintained.

But each call to Runner.run() is a fresh start.

Let's see that:

In [ ]:
agent = Agent(name="Assistant", model="gpt-5.4-mini")

In [ ]:
response = await Runner.run(agent, "Hi there. My name is Naresh")
print(response.final_output)

In [ ]:
response = await Runner.run(agent, "What's my name?")
print(response.final_output)

### Memory approach 1 - just manually pass in the list of dicts

In [ ]:
response = await Runner.run(agent, "Hi there. My name is Naresh.")
print(response.final_output)

In [ ]:
response.to_input_list()

In [ ]:
next_input = response.to_input_list() + [{"role": "user", "content": "What's my name?"}]
next_input

In [ ]:
response = await Runner.run(agent, next_input)
print(response.final_output)

### Another approach - use OpenAI Agents SDK built in SQLLite session

In [27]:
# This is created in-memory
# For an on-disk memory, use SQLiteSession("12345", "memory.db")

session = SQLiteSession("id-cts-1000")

In [28]:
response = await Runner.run(agent, "Hi there. My name is NC", session=session)
print(response.final_output)

Hi NC — nice to meet you. How can I help today?


In [29]:
response = await Runner.run(agent, "What's my name?", session=session)
print(response.final_output)

Your name is NC.
